# Akinator: sistema experto que adivina animales y países

El agente decide **qué preguntar** para adivinar en qué animal o país está pensando el usuario. El usuario responde *sí*, *no* o *no sé*.

Este notebook:

1. carga una instancia del entorno (la base de conocimiento);
2. muestra el estado del agente;
3. aplica a mano el encadenamiento hacia adelante, el encadenamiento hacia atrás y la selección por heurística;
4. juega una partida automática con cada método;
5. compara los 4 métodos sobre la misma instancia.

**Recompensas:** +100 por acertar, −2 por cada pregunta y −25 por una adivinanza fallida. Límite de 20 preguntas.

## 1. Cargar una instancia

`cargar_base` lee los JSON de `datos/` y completa cada candidato: lo que no se define es `False` (mundo cerrado) y se aplican las reglas hacia adelante para deducir atributos.

In [1]:
from base_conocimiento import cargar_base

base = cargar_base("animales")
print("Modo:", base.modo)
print("Candidatos:", len(base.candidatos))
print("Reglas:", len(base.reglas))
print("Preguntas:", len(base.preguntas))

ejemplo = "pinguino"
verdaderos = [atributo for atributo, valor in base.candidatos[ejemplo].items() if valor]
print(f"\nAtributos verdaderos de {ejemplo}:", verdaderos)

print("\nTres reglas:")
for regla in base.reglas[:3]:
    print(" ", regla["id"], "si", regla["si"], "->", regla["entonces"])

print("\nTres preguntas:")
for atributo in list(base.preguntas)[:3]:
    print(" ", atributo, "->", base.preguntas[atributo])

Modo: animales
Candidatos: 35
Reglas: 15
Preguntas: 71

Atributos verdaderos de pinguino: ['es_ser_vivo', 'tiene_plumas', 'pone_huevos', 'vive_en_agua', 'vive_en_frio', 'es_carnivoro', 'es_ave', 'es_ave_no_voladora', 'es_ave_polar']

Tres reglas:
  A1 si ['tiene_plumas'] -> es_ave
  A2 si ['da_leche'] -> es_mamifero
  A3 si ['tiene_escamas'] -> es_reptil

Tres preguntas:
  es_ser_vivo -> ¿Lo que piensas es un ser vivo?
  tiene_pelo -> ¿Lo que piensas tiene pelo?
  tiene_plumas -> ¿Lo que piensas tiene plumas?


## 2. El estado del agente

`Estado` guarda lo que el agente sabe: los hechos conocidos, los candidatos que quedan, las preguntas hechas, el puntaje y las métricas de ejecución.

In [2]:
from agente import crear_estado

estado = crear_estado(base)
print("hechos_conocidos:", estado.hechos_conocidos)
print("candidatos:", len(estado.candidatos), "->", estado.candidatos[:6], "...")
print("preguntas_hechas:", estado.preguntas_hechas)
print("puntos:", estado.puntos, "| límite de preguntas:", estado.limite_preguntas)
print("metricas:", estado.metricas)

hechos_conocidos: {}
candidatos: 35 -> ['perro', 'gato', 'vaca', 'caballo', 'cerdo', 'oveja'] ...
preguntas_hechas: []
puntos: 0 | límite de preguntas: 20
metricas: {'reglas_evaluadas': 0, 'reglas_disparadas': 0, 'hechos_inferidos': 0, 'nodos_generados': 0, 'hipotesis_probadas': 0, 'max_candidatos': 35, 'max_hechos': 0, 'tiempo': 0.0}


## 3. Encadenamiento hacia adelante, a mano

Partimos de un hecho (`tiene_plumas = sí`) y aplicamos las reglas hasta que no se deduce nada nuevo. Después se descartan los candidatos que contradicen los hechos.

In [3]:
from motor_inferencia import encadenamiento_adelante, filtrar_candidatos

hechos = {"tiene_plumas": True}
hechos, disparadas, evaluadas = encadenamiento_adelante(hechos, base.reglas)
print("Hechos tras el encadenamiento:", hechos)
print("Reglas evaluadas:", len(evaluadas), "| disparadas:", [r["id"] for r in disparadas])
for regla in disparadas:
    print(" ", regla["id"], ":", " y ".join(regla["si"]), "->", regla["entonces"])

restantes, descartados = filtrar_candidatos(list(base.candidatos), base.candidatos, hechos)
print(f"\nQuedan {len(restantes)}:", restantes)
print(f"Descartados {len(descartados)}:", descartados[:6], "...")

Hechos tras el encadenamiento: {'tiene_plumas': True, 'es_ave': True}
Reglas evaluadas: 29 | disparadas: ['A1']
  A1 : tiene_plumas -> es_ave

Quedan 7: ['pinguino', 'aguila', 'loro', 'gallina', 'avestruz', 'buho', 'flamenco']
Descartados 28: ['perro', 'gato', 'vaca', 'caballo', 'cerdo', 'oveja'] ...


## 4. Encadenamiento hacia atrás, a mano

Con pocos candidatos, el agente toma la **hipótesis** más probable y busca el atributo que mejor la separa de los demás. Si ese atributo se deduce con una regla, sigue la regla hasta algo que sí se pueda preguntar.

In [4]:
from agente import elegir_hipotesis
from motor_inferencia import encadenamiento_atras, explicar_atras

estado = crear_estado(base)
estado.candidatos = ["pinguino", "avestruz", "gallina"]
hipotesis = elegir_hipotesis(estado)
atributo = encadenamiento_atras(hipotesis, estado, base)

print("Candidatos:", estado.candidatos)
print("Hipótesis:", hipotesis)
print("Atributo que pide para confirmarla:", atributo)
print("Pregunta:", base.preguntas[atributo])
print("Explicación:", explicar_atras(hipotesis, estado, base))

Candidatos: ['pinguino', 'avestruz', 'gallina']
Hipótesis: avestruz
Atributo que pide para confirmarla: es_carnivoro
Pregunta: ¿Lo que piensas come carne?
Explicación: {'hipotesis': 'avestruz', 'objetivo': 'es_carnivoro', 'regla': None, 'separa': 2}


## 5. Heurísticas: *mitad* y *entropía*

Sobre los 35 animales calculamos cuántos responderían *sí* y cuántos *no* a cada pregunta. La heurística **mitad** busca el menor `|sí − no|`; la de **entropía** busca la mayor entropía.

In [5]:
from agente import atributos_utiles
from heuristica import contar_division, entropia, valores_atributos

estado = crear_estado(base)
print(f"{'atributo':<24}{'sí':>4}{'no':>4}{'|sí-no|':>9}{'entropía':>10}")
for atributo in atributos_utiles(estado, base)[:8]:
    si, no = contar_division(atributo, estado.candidatos, base.candidatos)
    print(f"{atributo:<24}{si:>4}{no:>4}{abs(si - no):>9}{entropia(si, no):>10.3f}")

print("\nMejores 3 según mitad:   ", valores_atributos(estado, base, "mitad")[:3])
print("Mejores 3 según entropía:", valores_atributos(estado, base, "entropia")[:3])

atributo                  sí  no  |sí-no|  entropía
tiene_pelo                18  17        1     0.999
tiene_plumas               7  28       21     0.722
tiene_escamas              3  32       29     0.422
tiene_aletas               5  30       25     0.592
da_leche                  21  14        7     0.971
pone_huevos               14  21        7     0.971
vuela                      5  30       25     0.592
vive_en_agua               9  26       17     0.822

Mejores 3 según mitad:    [('tiene_pelo', 1), ('tiene_cuatro_patas', 1), ('es_herbivoro', 5)]
Mejores 3 según entropía: [('tiene_pelo', 0.9994110647387553), ('tiene_cuatro_patas', 0.9994110647387553), ('es_herbivoro', 0.9852281360342516)]


**¿Por qué dan lo mismo?** Con preguntas de sí/no y candidatos igual de probables, tanto `|sí − no|` como la entropía dependen solo de la fracción `sí / (sí + no)`. Minimizar la diferencia equivale a acercar esa fracción a 1/2, que es justo donde la entropía es máxima. Por eso eligen la misma pregunta. Para que se comporten distinto haría falta darle un peso (popularidad) a cada candidato.

## 6. Una partida automática por método

`jugar_automatico` juega una partida completa contra un secreto conocido y devuelve la `Partida` con su **historial** paso a paso.

In [6]:
from automatico import jugar_automatico
from comparacion import METODOS

for nombre, metodo, heuristica, _ in METODOS:
    partida = jugar_automatico(base, "pinguino", metodo, heuristica, 20)
    print(f"=== {nombre} ===")
    for paso in partida.historial:
        print(f"  paso {paso['numero']}: {paso['tipo']:<9} {str(paso['atributo'] or ''):<22}"
              f"{str(paso['respuesta'] or ''):<6} candidatos={len(paso['candidatos']):<3} "
              f"puntos={paso['puntos']}")
    resumen = partida.historial[-1]["resumen"]
    print(f"  -> acertó: {resumen['encontro']} | preguntas: {resumen['preguntas']} "
          f"| puntaje: {resumen['puntaje']} | óptima: {resumen['optima']}\n")

=== Heurística mitad ===
  paso 0: inicio                                 candidatos=35  puntos=0
  paso 1: pregunta  tiene_pelo            no     candidatos=17  puntos=-2
  paso 2: pregunta  vive_en_agua          si     candidatos=8   puntos=-4
  paso 3: pregunta  tiene_aletas          no     candidatos=3   puntos=-6
  paso 4: pregunta  tiene_plumas          si     candidatos=2   puntos=-8
  paso 5: pregunta  vuela                 no     candidatos=1   puntos=-10
  paso 6: adivinar                               candidatos=1   puntos=90
  paso 7: fin                                    candidatos=1   puntos=90
  -> acertó: True | preguntas: 5 | puntaje: 90 | óptima: True

=== Heurística entropía ===
  paso 0: inicio                                 candidatos=35  puntos=0
  paso 1: pregunta  tiene_pelo            no     candidatos=17  puntos=-2
  paso 2: pregunta  vive_en_agua          si     candidatos=8   puntos=-4
  paso 3: pregunta  tiene_aletas          no     candidatos=3   puntos=

## 7. Comparación de los 4 métodos

`comparar` juega cada método contra **todos** los candidatos de la instancia y promedia las métricas. Se compara con un usuario sin errores y con uno que responde *no sé* el 30 % de las veces.

In [7]:
from comparacion import comparar


def imprimir(filas):
    print(f"{'Método':<26}{'% acierto':>10}{'Preguntas':>10}{'% óptima':>10}{'Tiempo ms':>10}"
          f"{'Nodos':>8}{'Reglas ev.':>11}{'Puntaje':>9}")
    for f in filas:
        print(f"{f['metodo']:<26}{f['aciertos']:>10.1f}{f['preguntas']:>10.2f}{f['optimas']:>10.1f}"
              f"{f['tiempo_ms']:>10.2f}{f['nodos_generados']:>8.1f}{f['reglas_evaluadas']:>11.1f}"
              f"{f['puntaje']:>9.1f}")


secretos = list(base.candidatos)
print("Usuario sin errores")
imprimir(comparar(base, secretos, 20))
print("\nUsuario que responde 'no sé' el 30 % de las veces")
imprimir(comparar(base, secretos, 20, prob_no_se=0.3, repeticiones=5))

Usuario sin errores
Método                     % acierto Preguntas  % óptima Tiempo ms   Nodos Reglas ev.  Puntaje
Heurística mitad               100.0      5.20     100.0      0.48    85.4       84.2     89.6
Heurística entropía            100.0      5.20     100.0      0.49    85.4       84.2     89.6
Encadenamiento adelante        100.0      6.26      54.3      0.46    98.0      101.0     87.5
Encadenamiento atrás           100.0      6.09      54.3      0.56   100.6      105.3     87.8

Usuario que responde 'no sé' el 30 % de las veces


Método                     % acierto Preguntas  % óptima Tiempo ms   Nodos Reglas ev.  Puntaje
Heurística mitad               100.0      7.20      38.3      0.62   119.5       80.1     83.5
Heurística entropía            100.0      7.20      37.7      0.66   119.7       79.8     83.6
Encadenamiento adelante        100.0      8.91      26.3      0.64   139.6       99.6     79.6
Encadenamiento atrás           100.0      8.23      26.3      0.77   148.2       94.9     79.1


## Conclusión

- Los cuatro métodos adivinan siempre cuando el usuario no se equivoca. Cambia cuántas preguntas necesitan.
- Las heurísticas *mitad* y *entropía* hacen menos preguntas, porque cada respuesta parte a los candidatos casi por la mitad.
- El encadenamiento hacia adelante pregunta en un orden fijo, sin elegir la mejor pregunta, y necesita más.
- El encadenamiento hacia atrás sigue una hipótesis y confirma lo necesario: razonable con pocos candidatos, pero con muchos hace más preguntas que las heurísticas.
- Con *no sé* todos pierden eficiencia.